In [1]:
import os, glob, json, warnings
from tqdm import tqdm
import numpy as np
import networkx as nx
import pandas as pd

from src.cell_graph_functions import positive_indices_and_labels

In [2]:
# ==============================
# Graph I/O (from preprocessed NPZ)
# ==============================
def load_cell_graph_from_npz(data: np.lib.npyio.NpzFile) -> nx.Graph:
    """
    Reconstruct a NetworkX graph from edges and node count stored in a preprocessed NPZ file.
    Expects:
      - 'graph_edges'   : (E, 2) int array of undirected cell adjacencies (labels indexed 0..N-1 or any set)
      - 'graph_n_nodes' : scalar int (number of cells)
    """
    if "graph_edges" not in data.files or "graph_n_nodes" not in data.files:
        raise KeyError("NPZ must contain 'graph_edges' and 'graph_n_nodes'.")

    edges = np.asarray(data["graph_edges"]).reshape(-1, 2)
    n_nodes = int(data["graph_n_nodes"])

    G = nx.Graph()
    G.add_nodes_from(range(n_nodes))
    if edges.size:
        G.add_edges_from(edges.tolist())
    return G

# ==============================
# Curvature proxy from single HKS time
# ==============================
def hks_to_curv_proxy_single_t(hks_col, t, d=2):
    """
    Convert a single HKS column (N,) at diffusion time t to a curvature-like target:
      Y_t = (6/t) * [ (4πt)^{d/2} * HKS_t - 1 ]  ~  Gaussian curvature
    Returns float32 (N,).
    """
    h = np.asarray(hks_col, dtype=np.float64).reshape(-1)
    t = float(t)
    Y = (6.0 / t) * ((4.0 * np.pi * t) ** (d / 2.0) * h - 1.0)
    return Y.astype(np.float32)

# ==============================
# Marker filtering
# ==============================
def filter_markers(marker_names, markers_array, exclude_names):
    """
    Drop markers listed in `exclude_names` from both names and (N,M) array.
    Returns (filtered_names, filtered_markers).
    """
    if not exclude_names:
        return list(marker_names), np.asarray(markers_array, dtype=np.float32)
    keep_idx = [i for i, n in enumerate(marker_names) if n not in exclude_names]
    if not keep_idx:
        raise ValueError("After excluding markers, no markers remain. Adjust `exclude_markers`.")
    names = [marker_names[i] for i in keep_idx]
    X = np.asarray(markers_array, dtype=np.float32)[:, keep_idx]
    return names, X

# ==============================
# Export (Project B format)
# ==============================
def _edges_from_graph(organoid_graph: nx.Graph, N_expected: int) -> np.ndarray:
    """
    Convert NetworkX graph to an undirected edge array (E,2) with u < v (int64).
    Validates node set size == N_expected.
    """
    nodes = sorted(organoid_graph.nodes())
    if len(nodes) != N_expected:
        # If nodes are labels (0..N-1) but not consecutive, reindex
        id_map = {old: new for new, old in enumerate(nodes)}
        N = len(nodes)
        warnings.warn(f"Graph node count {len(nodes)} != N={N_expected}; reindexing nodes 0..{N-1}.")
    else:
        id_map = None
        N = N_expected

    edges_uplow = set()
    for u, v in organoid_graph.edges():
        uu = id_map[u] if id_map is not None else u
        vv = id_map[v] if id_map is not None else v
        if uu == vv:
            continue
        a, b = (uu, vv) if uu < vv else (vv, uu)
        edges_uplow.add((int(a), int(b)))

    if not edges_uplow:
        return np.zeros((0, 2), dtype=np.int64)
    arr = np.array(sorted(edges_uplow), dtype=np.int64)
    # sanity
    if (arr[:, 0] < 0).any() or (arr[:, 1] >= N).any():
        raise ValueError("Edge indices out of range after reindex.")
    return arr

def export_organoid_npz(
    out_dir: str,
    organoid_id: str,
    y: np.ndarray,                   # (N,) float32
    x_bin: np.ndarray,               # (N, M) float32, binary 0/1
    organoid_graph: nx.Graph,
    marker_names=None,               # optional list[str] length M
    aux_meta: dict | None = None,    # OPTIONAL: {'total_area':..., 'total_volume':..., 'complexity':...}
):
    """
    Save one organoid in the streamlined Project (B) format:
      - x         : (N, M) float32 (binary)
      - y         : (N,)   float32
      - edges     : (E, 2) int64 (u < v)
      - N, M      : int64 scalars
    Sidecars:
      - <file>_markers.json : marker_names (if provided)
      - <file>_tvalues.json : [t_value]    (if provided)
      - <file>_aux.json     : { 'organoid_id', 'total_area', 'total_volume', 'complexity', 't_value' }
    """
    os.makedirs(out_dir, exist_ok=True)

    # Validate & coerce x,y
    X = np.asarray(x_bin, dtype=np.float32)
    if X.ndim != 2:
        raise ValueError(f"x must be 2-D (N,M); got {X.shape}")
    # Ensure {0,1}
    if not np.isin(X, [0.0, 1.0]).all():
        X = (X > 0.5).astype(np.float32)

    y = np.asarray(y, dtype=np.float32).reshape(-1)
    if y.ndim != 1:
        raise ValueError(f"y must be 1-D (N,); got {y.shape}")

    N, M = X.shape
    if y.shape[0] != N:
        raise ValueError(f"y length {y.shape[0]} != N={N}")
    if N <= 1:
        raise ValueError("N must be > 1 (need at least 2 cells).")
    if not np.isfinite(y).all():
        raise ValueError("y contains NaN/Inf.")

    # Build edges
    edge_array = _edges_from_graph(organoid_graph, N_expected=N)

    # Save NPZ (numeric only)
    file_stem = f"organoid_{organoid_id}"
    out_path = os.path.join(out_dir, f"{file_stem}.npz")
    np.savez_compressed(
        out_path,
        x=X,
        y=y,
        edges=edge_array,
        N=np.int64(N),
        M=np.int64(M),
    )

    # Sidecars
    if marker_names is not None:
        with open(os.path.join(out_dir, f"{file_stem}_markers.json"), "w") as f:
            json.dump(list(marker_names), f, indent=2)
    if aux_meta:
        save_aux_metadata(os.path.join(out_dir, file_stem), organoid_id, aux_meta)


    return out_path


def save_aux_metadata(sidecar_path_no_ext: str, organoid_id: str, aux: dict) -> None:
    """
    Save auxiliary metadata as JSON next to the NPZ. Uses a distinct filename to avoid
    interfering with the GNN loader. Only writes if `aux` is non-empty.

    Example output file: <out_dir>/organoid_<id>_aux.json
    """
    if not aux:
        return
    payload = {
        "organoid_id": str(organoid_id),
        **aux
    }
    json_path = f"{sidecar_path_no_ext}_aux.json"
    with open(json_path, "w") as f:
        json.dump(payload, f, indent=2)

In [3]:
# ==============================
# CONFIG (edit here)
# ==============================
save_dir = "../GraphNN/training_data"      # output dir for NPZs
include_timepoints = ["day4", "day4p5", "day4p5-more"]  # or None for all
exclude_markers = {}             # markers to drop
selected_t_index = 0                                    # which HKS column to use
selected_t_value = 1.0                                  # its diffusion time t

# Preprocessed base dir (Project A)
preproc_base = os.path.join("..", "NicoleData", "preprocessed")

tsne_df = pd.read_csv("../NicoleData/tsne_results.csv")
# Expecting columns: label_uid (e.g. "day3p5_A01_42"), ..., complexity (4th column)
complexity_dict = dict(zip(tsne_df["label_uid"], tsne_df["complexity"]))
print(f"Loaded {len(complexity_dict)} complexity scores")

# ==============================
# Discover timepoints
# ==============================
def _choose_timepoints(root, include):
    all_tps = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    return all_tps if include is None else [tp for tp in all_tps if tp in include]

timepoints = _choose_timepoints(preproc_base, include_timepoints)
print(f"Using {len(timepoints)} timepoints: {timepoints}")

# ==============================
# MAIN LOOP
# ==============================
total_exported, total_skipped = 0, 0

for tp in timepoints:
    print(f"\n=== Exporting timepoint: {tp} ===")
    tp_dir = os.path.join(preproc_base, tp)
    npz_files = sorted(glob.glob(os.path.join(tp_dir, "*.npz")))
    if not npz_files:
        print(f"  ⚠ No .npz files found for {tp}, skipping.")
        continue

    exported, skipped = 0, 0

    for fpath in tqdm(npz_files, desc=f"{tp} (export)"):
        try:
            data = np.load(fpath, allow_pickle=True)
        except Exception as e:
            print(f"  Could not load {os.path.basename(fpath)}: {e}")
            skipped += 1
            continue

        # IDs
        well = str(data["well"])
        organoid_id = str(data["organoid_id"])
        label_uid = f"{tp}_{well}_{organoid_id}"

        # Arrays
        hks_cell = np.asarray(data["fields_cell"], dtype=np.float32)   # (N, T)
        markers_cell = np.asarray(data["markers_cell"], dtype=np.float32)  # (N, M_raw)
        marker_names = list(data["marker_names"])

        # Graph
        try:
            cell_graph = load_cell_graph_from_npz(data)
        except Exception as e:
            print(f"  {label_uid}: bad graph: {e} — skipping.")
            skipped += 1
            continue

        # Basic shape checks
        if hks_cell.ndim != 2 or selected_t_index >= hks_cell.shape[1]:
            print(f"  {label_uid}: invalid HKS shape {hks_cell.shape} for t-index {selected_t_index}, skipping.")
            skipped += 1
            continue
        if markers_cell.ndim != 2 or markers_cell.shape[0] != hks_cell.shape[0]:
            print(f"  {label_uid}: markers shape {markers_cell.shape} incompatible with HKS {hks_cell.shape}, skipping.")
            skipped += 1
            continue

        # Filter markers
        marker_names_f, X = filter_markers(marker_names, markers_cell, exclude_markers)

        # Binarize markers
        _, X = positive_indices_and_labels(X)
        X = X.astype(np.float32)

        # Build y from selected HKS column
        y = hks_to_curv_proxy_single_t(hks_cell[:, selected_t_index], t=selected_t_value)  # (N,)

        # Final guardrails
        N = X.shape[0]
        if y.shape[0] != N or N <= 1:
            print(f"  {label_uid}: invalid N or y length (N={N}, len(y)={y.shape[0]}), skipping.")
            skipped += 1
            continue
        if not np.isfinite(y).all():
            print(f"  {label_uid}: y has NaN/Inf, skipping.")
            skipped += 1
            continue

        # Build axiliary metadata
        aux_meta = {}
        aux_meta["surface_area"] = float(np.sum(data["vertex_areas"]))
        aux_meta["volume"] = float(data["volume"])
        aux_meta["t_value"] = float(selected_t_value)

        complexity_value = complexity_dict.get(label_uid, np.nan)
        aux_meta["complexity"] = float(complexity_value)
        
        # Export
        try:
            export_organoid_npz(
                out_dir=save_dir,
                organoid_id=label_uid,     # string ID goes into filename
                y=y,                       # (N,)
                x_bin=X,                   # (N, M_filtered)
                organoid_graph=cell_graph,
                marker_names=marker_names_f,
                aux_meta=aux_meta,
            )
            exported += 1
        except Exception as e:
            print(f"  {label_uid}: export failed: {e}")
            skipped += 1

    total_exported += exported
    total_skipped  += skipped
    print(f"  Done {tp}: exported={exported}, skipped={skipped}")

print(f"\nAll timepoints done. Total exported={total_exported}, skipped={total_skipped}")


Loaded 2668 complexity scores
Using 3 timepoints: ['day4', 'day4p5', 'day4p5-more']

=== Exporting timepoint: day4 ===


day4 (export): 100%|██████████| 357/357 [00:01<00:00, 273.26it/s]


  Done day4: exported=357, skipped=0

=== Exporting timepoint: day4p5 ===


day4p5 (export): 100%|██████████| 109/109 [00:00<00:00, 220.73it/s]


  Done day4p5: exported=109, skipped=0

=== Exporting timepoint: day4p5-more ===


day4p5-more (export):  75%|███████▍  | 398/534 [00:01<00:00, 260.82it/s]

  day4p5-more_C05_152: invalid N or y length (N=1, len(y)=1), skipping.


day4p5-more (export): 100%|██████████| 534/534 [00:02<00:00, 256.98it/s]

  Done day4p5-more: exported=533, skipped=1

All timepoints done. Total exported=999, skipped=1
